# Create Lakehouse Views

**Run with Bronze_Lakehouse set as the default lakehouse.**

Creates:
- `vw_player_performance_enriched` — deduped base view
- `vw_player_performance_vs_expected` — per-player baselines + vs-expected deltas for all metrics

Run cells top to bottom — Block 2 depends on Block 1.

## Block 1 · vw_player_performance_enriched

Deduplicates `Event History` on `(dg_id, event_id, year, round)`, keeps latest `event_completed` when duplicates exist, and adds `score_vs_par`.

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_player_performance_enriched AS
WITH ranked AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY dg_id, event_id, year, round
            ORDER BY event_completed DESC
        ) AS _rn
    FROM `Event History`
    WHERE dg_id IS NOT NULL
      AND round  IS NOT NULL
)
SELECT
    -- Identity / event
    tour,
    year,
    season,
    event_id,
    CAST(event_completed AS DATE)  AS event_completed,
    event_name,
    dg_id,
    player_name,
    fin_text,

    -- Course
    course_num,
    course_name,
    course_par,

    -- Round metadata
    round,
    score,
    start_hole,
    CAST(score AS INT) - CAST(course_par AS INT)  AS score_vs_par,

    -- Scoring breakdown
    eagles_or_better,
    birdies,
    pars,
    bogies,
    doubles_or_worse,

    -- Strokes gained
    sg_ott,
    sg_app,
    sg_arg,
    sg_putt,
    sg_t2g,
    sg_total,

    -- Traditional stats (populated when traditional_stats = 'yes')
    driving_acc,
    driving_dist,
    gir,
    scrambling,
    prox_fw,
    prox_rgh,
    great_shots,
    poor_shots,

    -- Data availability flags
    sg_categories,
    traditional_stats

FROM ranked
WHERE _rn = 1
""")

print("vw_player_performance_enriched created.")

## Block 2 · vw_player_performance_vs_expected

Computes each player's career-average baseline per metric, then joins back to produce `_vs_expected` delta columns.

- SG baselines: rounds where `sg_categories = 'yes'` only
- Traditional stat baselines: rounds where `traditional_stats = 'yes'` only
- Positive delta = outperformed own baseline; negative = underperformed

In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW vw_player_performance_vs_expected AS
WITH player_baselines AS (
    SELECT
        dg_id,

        -- SG baselines
        AVG(CASE WHEN sg_categories    = 'yes' THEN sg_ott   END) AS baseline_sg_ott,
        AVG(CASE WHEN sg_categories    = 'yes' THEN sg_app   END) AS baseline_sg_app,
        AVG(CASE WHEN sg_categories    = 'yes' THEN sg_arg   END) AS baseline_sg_arg,
        AVG(CASE WHEN sg_categories    = 'yes' THEN sg_putt  END) AS baseline_sg_putt,
        AVG(CASE WHEN sg_categories    = 'yes' THEN sg_t2g   END) AS baseline_sg_t2g,
        AVG(CASE WHEN sg_categories    = 'yes' THEN sg_total END) AS baseline_sg_total,

        -- Traditional stat baselines
        AVG(CASE WHEN traditional_stats = 'yes' THEN driving_acc  END) AS baseline_driving_acc,
        AVG(CASE WHEN traditional_stats = 'yes' THEN driving_dist END) AS baseline_driving_dist,
        AVG(CASE WHEN traditional_stats = 'yes' THEN gir          END) AS baseline_gir,
        AVG(CASE WHEN traditional_stats = 'yes' THEN scrambling   END) AS baseline_scrambling,
        AVG(CASE WHEN traditional_stats = 'yes' THEN prox_fw      END) AS baseline_prox_fw,
        AVG(CASE WHEN traditional_stats = 'yes' THEN prox_rgh     END) AS baseline_prox_rgh,
        AVG(CASE WHEN traditional_stats = 'yes' THEN great_shots  END) AS baseline_great_shots,
        AVG(CASE WHEN traditional_stats = 'yes' THEN poor_shots   END) AS baseline_poor_shots

    FROM vw_player_performance_enriched
    GROUP BY dg_id
)
SELECT
    e.*,

    -- SG vs player career baseline
    e.sg_ott   - b.baseline_sg_ott   AS sg_ott_vs_expected,
    e.sg_app   - b.baseline_sg_app   AS sg_app_vs_expected,
    e.sg_arg   - b.baseline_sg_arg   AS sg_arg_vs_expected,
    e.sg_putt  - b.baseline_sg_putt  AS sg_putt_vs_expected,
    e.sg_t2g   - b.baseline_sg_t2g   AS sg_t2g_vs_expected,
    e.sg_total - b.baseline_sg_total AS sg_total_vs_expected,

    -- Traditional stats vs player career baseline
    e.driving_acc  - b.baseline_driving_acc  AS driving_acc_vs_expected,
    e.driving_dist - b.baseline_driving_dist AS driving_dist_vs_expected,
    e.gir          - b.baseline_gir          AS gir_vs_expected,
    e.scrambling   - b.baseline_scrambling   AS scrambling_vs_expected,
    e.prox_fw      - b.baseline_prox_fw      AS prox_fw_vs_expected,
    e.prox_rgh     - b.baseline_prox_rgh     AS prox_rgh_vs_expected,
    e.great_shots  - b.baseline_great_shots  AS great_shots_vs_expected,
    e.poor_shots   - b.baseline_poor_shots   AS poor_shots_vs_expected,

    -- Baselines exposed for diagnostics / downstream use
    b.baseline_sg_ott,
    b.baseline_sg_app,
    b.baseline_sg_arg,
    b.baseline_sg_putt,
    b.baseline_sg_t2g,
    b.baseline_sg_total,
    b.baseline_driving_acc,
    b.baseline_driving_dist,
    b.baseline_gir,
    b.baseline_scrambling,
    b.baseline_great_shots,
    b.baseline_poor_shots

FROM vw_player_performance_enriched e
LEFT JOIN player_baselines b ON e.dg_id = b.dg_id
""")

print("vw_player_performance_vs_expected created.")

## Verify

In [ ]:
enriched = spark.sql("SELECT COUNT(*) AS rows FROM vw_player_performance_enriched").collect()[0][0]
vs_exp   = spark.sql("SELECT COUNT(*) AS rows FROM vw_player_performance_vs_expected").collect()[0][0]
raw      = spark.sql("SELECT COUNT(*) AS rows FROM `Event History`").collect()[0][0]

print(f"Event History (raw):            {raw:,}")
print(f"vw_player_performance_enriched: {enriched:,}  (deduped)")
print(f"vw_player_performance_vs_expected: {vs_exp:,}")

# Quick sanity check on vs_expected
spark.sql("""
    SELECT player_name, ROUND(baseline_sg_total, 3) AS baseline_sg_total,
           ROUND(AVG(sg_total_vs_expected), 3) AS avg_sg_total_vs_exp
    FROM vw_player_performance_vs_expected
    WHERE sg_categories = 'yes'
    GROUP BY player_name, baseline_sg_total
    ORDER BY baseline_sg_total DESC
    LIMIT 10
""").show(truncate=False)